## 리랭커로 검색된 문서 순위  조정하기
### 교차 인코더 리랭커
- 검색 결과를 관련성이 높은 순으로 재정렬.


In [1]:
# !uv add sentence-transformers

Resolved 140 packages in 1.42s
   Building rag-two @ file:///D:/skc0902/rag_one_2/rag_two
      Built rag-two @ file:///D:/skc0902/rag_one_2/rag_two
 Downloaded tokenizers
 Downloaded hf-xet
 Downloaded sympy
 Downloaded scikit-learn
 Downloaded scipy
 Downloaded transformers
 Downloaded torch
Prepared 21 packages in 56.60s
Uninstalled 1 package in 35ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 21 packages in 15.05s
 + annotated-doc==0.0.5
 + filelock==4.0.1
 + hf-xet==1.6.0
 + huggingface-hub==1.32.0
 + markdown-it-py==4.2.0
 + mdurl==0.1.2
 + mpmath==1.3.0
 + narwhals==2.26.0
 ~ rag-two==0.1.0 (from file:///D:/skc0902/rag_one_2/rag_two)
 + rich==15.0.0
 + safetensors==0.8.0
 + scikit-learn==1.9.1
 + scipy==1.18.1
 + sentence-transformers==6.1.0
 + shellingham==1.5.4
 + sympy==1.14.0
 + threadpo

In [2]:
# ---------------------------------------------------------------------------
# HuggingFaceEmbeddings시 오류방지
# OSError: sentence-transformers/msmarco-distilbert-dot-v5 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
# If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
# ---------------------------------------------------------------------------

import os

# 저장된 토큰의 자동 전송 방지 — 라이브러리 import 전에 설정
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"


In [4]:
# Jupyter Notebook용 진행률 표시바 설치
# !uv add ipywidgets jupyter

Resolved 214 packages in 959ms
   Building rag-two @ file:///D:/skc0902/rag_one_2/rag_two
      Built rag-two @ file:///D:/skc0902/rag_one_2/rag_two
 Downloaded pywinpty
 Downloaded widgetsnbextension
 Downloaded notebook
 Downloaded babel
 Downloaded jupyterlab
Prepared 51 packages in 12.23s
Uninstalled 1 package in 2ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 51 packages in 7.69s
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==26.1.0
 + arrow==1.4.0
 + async-lru==2.3.0
 + babel==2.18.0
 + bleach==6.4.0
 + cffi==2.1.1
 + fastjsonschema==2.22.2
 + fqdn==1.5.1
 + ipywidgets==8.1.9
 + isoduration==20.11.0
 + json5==0.15.0
 + jsonschema==4.26.0
 + jsonschema-specifications==2025.9.1
 + jupyter==1.1.1
 + jupyter-builder==1.2.3
 + jupyter-console==6.6.3
 + jupyter-events==0.12.1
 + jupyter-lsp==2.3.1

In [6]:
# Sentence-Transformers 라이브러리에서
# 문장을 임베딩(Vector)으로 변환하는 SentenceTransformer 클래스 import
from sentence_transformers import SentenceTransformer


# 사전 학습된 문장 임베딩 모델 로드
model = SentenceTransformer(

    # MS MARCO 검색 데이터셋 기반으로 학습된
    # DistilBERT 계열 임베딩 모델
    "sentence-transformers/msmarco-distilbert-dot-v5",

    # Hugging Face에 저장된 인증 토큰을 자동으로 사용하지 않도록 설정
    token=False,

    # 내부 Transformer 모델을 불러올 때도
    # Hugging Face 인증 토큰을 사용하지 않도록 설정
    model_kwargs={"token": False},

    # tokenizer / processor를 불러올 때도
    # 인증 토큰을 사용하지 않도록 설정
    processor_kwargs={"token": False},

    # 모델 설정(config)을 불러올 때도
    # 인증 토큰을 사용하지 않도록 설정
    config_kwargs={"token": False},
)


# "Hello world" 문장을 임베딩 벡터로 변환
# encode() 결과의 크기(shape)를 출력
#
# 예상 결과:
# (1, 768)
#
# 1   : 입력 문장 개수
# 768 : 하나의 문장을 표현하는 임베딩 벡터 차원
print(model.encode(["Hello world"]).shape)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1516.76it/s]


(1, 768)


In [7]:
# 문서 목록(docs)의 내용을 보기 좋게 출력하기 위한 함수
def pretty_print_docs(docs):

    # 여러 문서를 하나의 문자열로 합쳐서 출력
    print(

        # 각 문서 사이에 구분선(하이픈 100개)을 추가
        f"\n{'-' * 100}\n".join(

            # docs에 들어있는 각 문서를 순서대로 반복
            [
                # 문서 번호를 1부터 표시하고,
                # 각 Document 객체의 실제 텍스트 내용(page_content)을 출력
                f"Document {i+1}:\n\n" + d.page_content

                # enumerate()를 사용해
                # i에는 문서의 인덱스(0, 1, 2, ...)
                # d에는 각 Document 객체를 저장
                for i, d in enumerate(docs)
            ]
        )
    )

In [8]:
# ##### TEST1 ::: 허깅페이스 모델 다운로드 확인 코드 #####

# Hugging Face Hub에서 특정 파일을 다운로드하기 위한 함수 import
# hf_hub_download()는 Hugging Face 저장소(repo) 안의 파일 1개를 다운로드할 때 사용
from huggingface_hub import hf_hub_download


# Hugging Face 저장소에서 config.json 파일 다운로드
path = hf_hub_download(

#     # 다운로드할 모델 저장소 이름
#     # 여기서는 Sentence-Transformers의 MS MARCO 기반 임베딩 모델 사용
    repo_id="sentence-transformers/msmarco-distilbert-dot-v5",

#     # 저장소 안에서 다운로드할 파일 이름
#     # config.json은 모델 구조와 설정 정보를 담고 있는 파일
    filename="config.json",

#     # Hugging Face 인증 토큰을 사용하지 않도록 설정
#     # 공개 모델이므로 토큰 없이 다운로드 가능
    token=False,
)


# 다운로드된 파일이 로컬 PC의 어느 경로에 저장되었는지 출력
# 보통 Hugging Face 캐시 디렉터리 경로가 출력됨
print(path)

C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--msmarco-distilbert-dot-v5\snapshots\c53e81ab2aa9dae60c5e35f892daaf1d67704012\config.json


In [9]:
# LangChain Document 객체 목록(docs)을
# 보기 좋게 출력하기 위한 사용자 정의 함수
def pretty_print_docs(docs):

    # docs에 들어있는 문서를 하나씩 반복
    # start=1을 사용하므로 문서 번호가 1부터 시작
    for i, doc in enumerate(docs, start=1):

        # 현재 문서의 번호 출력
        print(f"문서 {i}")

        # Document 객체의 실제 텍스트 내용 출력
        # LangChain의 Document 객체는
        # page_content 속성에 본문 내용을 저장
        print(doc.page_content)

        # 문서와 문서 사이를 구분하기 위해
        # 하이픈(-) 80개 출력
        print("-" * 80)


# 함수 실행 예시
# docs에 저장된 모든 문서를 순서대로 출력
# 앞에 #이 있으므로 현재는 실행되지 않음
# pretty_print_docs(docs)

In [10]:
# ##### TEST2 ::: 허깅페이스 모델 정상작동 확인  #####
# # test 2

# Sentence-Transformers 라이브러리에서
# 문장 임베딩 모델을 불러오기 위한 SentenceTransformer 클래스 import
from sentence_transformers import SentenceTransformer


# Hugging Face에 공개된 사전 학습 임베딩 모델 로드
model = SentenceTransformer(

#     # MS MARCO 검색 데이터셋 기반으로 학습된
#     # DistilBERT 계열의 문장 임베딩 모델
    "sentence-transformers/msmarco-distilbert-dot-v5",

#     # Hugging Face 인증 토큰을 사용하지 않도록 설정
#     # 공개 모델이므로 일반적으로 토큰 없이도 다운로드 가능
    token=False,
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1242.07it/s]


In [15]:
# !uv add langchain-huggingface
# !uv add faiss-cpu

Resolved 216 packages in 271ms
   Building rag-two @ file:///D:/skc0902/rag_one_2/rag_two
      Built rag-two @ file:///D:/skc0902/rag_one_2/rag_two
 Downloaded faiss-cpu
Prepared 2 packages in 3.52s
Uninstalled 1 package in 1ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 104ms
 + faiss-cpu==1.15.1
 ~ rag-two==0.1.0 (from file:///D:/skc0902/rag_one_2/rag_two)


In [16]:
# RAG
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss

# 문서 로드
documents = TextLoader("../data/appendix-keywords.txt", encoding="utf-8").load()

# 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

# 문서 분할
texts = text_splitter.split_documents(documents)

# 임베딩 모델 설정
embeddingsModel = HuggingFaceEmbeddings(
    model_name="sentence-transformers/msmarco-distilbert-dot-v5"
)

# 문서로부터 FAISS 인덱스 생성 및 검색기 설정
retriever = FAISS.from_documents(texts, embeddingsModel).as_retriever(
    search_kwargs={"k": 10}
)

# 질의 설정
query = "Word2Vec 에 대해서 알려줄래?"

# 질의 수행 및 결과 문서 반환
docs = retriever.invoke(query)

# 결과 문서 출력
pretty_print_docs(docs)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3896.21it/s]


문서 1
Open Source

정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협업과 혁신을 촉진하는 데 중요한 역할을 합니다.
예시: 리눅스 운영 체제는 대표적인 오픈 소스 프로젝트입니다.
연관키워드: 소프트웨어 개발, 커뮤니티, 기술 협업

Structured Data

정의: 구조화된 데이터는 정해진 형식이나 스키마에 따라 조직된 데이터입니다. 이는 데이터베이스, 스프레드시트 등에서 쉽게 검색하고 분석할 수 있습니다.
예시: 관계형 데이터베이스에 저장된 고객 정보 테이블은 구조화된 데이터의 예입니다.
연관키워드: 데이터베이스, 데이터 분석, 데이터 모델링

Parser
--------------------------------------------------------------------------------
문서 2
정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.
예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.
연관키워드: 자연어 처리, 딥러닝, 텍스트 생성

FAISS (Facebook AI Similarity Search)

정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.
예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.
연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화

Open Source
--------------------------------------------------------------------------------
문서 3
InstructGPT

정의: InstructGPT는 사용자의 지시에 따라 특정한 작업을 수행하기 위해 최적화된 GPT 모델입니다. 이 모델은 보다 정

In [19]:
# 오류대비용
# import sys
# !{sys.executable} -m pip install -U sentence-transformers

# !uv add sentence-transformers
# !uv add langchain-classic

Resolved 216 packages in 1ms
Checked 193 packages in 5ms
Resolved 216 packages in 2ms
Checked 193 packages in 6ms


# Windows 설정에서 Developer Mode를 켜면 일반 사용자 권한에서도 symlink 사용이 가능해짐. 그냥 안함.
- Hugging Face가 같은 파일을 중복 저장하지 않도록 하는 방법인데, 테스트 용이므로 넘어감.

In [20]:
# 타입 힌트에 사용할 List, Tuple import
# List[float]             → 실수(float) 목록
# Tuple[str, str]         → 문자열 2개로 이루어진 튜플
from typing import List, Tuple


# Hugging Face / Sentence-Transformers의 CrossEncoder import
# 질문과 문서를 한 쌍으로 함께 입력해서 관련도 점수를 계산
from sentence_transformers import CrossEncoder


# LangChain의 ContextualCompressionRetriever import
# 기본 Retriever가 찾은 문서를 Compressor로 다시 압축/재정렬해서 반환
from langchain_classic.retrievers import ContextualCompressionRetriever


# Cross-Encoder 점수를 이용해 문서를 재정렬하는 Reranker import
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker


# LangChain에서 CrossEncoderReranker와 연결하기 위한
# CrossEncoder 기본 인터페이스(Base Class) import
from langchain_classic.retrievers.document_compressors.cross_encoder import BaseCrossEncoder


# 예전 방식 또는 별도 패키지의 HuggingFaceCrossEncoder import 예시
# 현재 코드에서는 직접 클래스를 만들어 사용하므로 주석 처리
# from langchain_community.cross_encoders import HuggingFaceCrossEncoder



# LangChain의 BaseCrossEncoder를 상속받아
# Sentence-Transformers의 CrossEncoder를 연결하는 사용자 정의 클래스
class MyHuggingFaceCrossEncoder(BaseCrossEncoder):

    # 클래스 생성자
    # model_name으로 사용할 CrossEncoder 모델 이름을 전달받음
    def __init__(self, model_name: str):

        # Sentence-Transformers의 CrossEncoder 모델 로드
        self.model = CrossEncoder(model_name)


    # 질문-문서 쌍들의 관련도 점수를 계산하는 메서드
    def score(
        self,

        # 예:
        # [
        #   ("질문", "문서1"),
        #   ("질문", "문서2"),
        #   ("질문", "문서3")
        # ]
        text_pairs: List[Tuple[str, str]]

    ) -> List[float]:

        # CrossEncoder.predict()를 이용해
        # 각 질문-문서 쌍의 관련도 점수 계산
        #
        # predict() 결과는 NumPy 배열 형태이므로
        # .tolist()를 사용해 Python List 형태로 변환
        return self.model.predict(text_pairs).tolist()



# ----------------------------------------------------
# 1. Cross-Encoder 모델 초기화
# ----------------------------------------------------

# BAAI의 bge-reranker-v2-m3 모델 사용
# 검색된 문서와 질문의 관련도를 다시 평가하는 Reranker 모델
model = MyHuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3"
)



# ----------------------------------------------------
# 2. Reranker 생성
# ----------------------------------------------------

# CrossEncoderReranker 생성
#
# model   : 위에서 만든 Cross-Encoder 모델
# top_n=3 : 재평가한 문서 중 점수가 높은 상위 3개만 선택
compressor = CrossEncoderReranker(
    model=model,
    top_n=3
)



# ----------------------------------------------------
# 3. ContextualCompressionRetriever 생성
# ----------------------------------------------------

# 기존 Retriever가 검색한 결과를
# CrossEncoderReranker로 다시 평가하는 Retriever 생성
compression_retriever = ContextualCompressionRetriever(

    # 검색된 문서를 재평가하고 압축하는 Reranker
    base_compressor=compressor,

    # 1차 검색을 담당하는 기존 Retriever
    # 예: FAISS, Chroma 등의 VectorStore Retriever
    base_retriever=retriever
)



# ----------------------------------------------------
# 4. 질문으로 문서 검색 + Reranking
# ----------------------------------------------------

# 질문 입력
#
# 처리 과정:
#
# 질문
#   ↓
# base_retriever
#   ↓
# 여러 후보 문서 검색
#   ↓
# CrossEncoderReranker
#   ↓
# 질문과 각 문서의 관련도 재평가
#   ↓
# 관련도 높은 상위 3개 선택
compressed_docs = compression_retriever.invoke(
    "Word2Vec 에 대해서 알려줄래?"
)



# ----------------------------------------------------
# 5. 최종 문서 출력
# ----------------------------------------------------

# 이전에 정의한 pretty_print_docs() 함수를 사용해
# Reranking된 상위 3개 문서를 보기 좋게 출력
pretty_print_docs(compressed_docs)

d:\skc0902\rag_one_2\rag_two\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 492.71it/s]


문서 1
Crawling

정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다.
예시: 구글 검색 엔진이 인터넷 상의 웹사이트를 방문하여 콘텐츠를 수집하고 인덱싱하는 것이 크롤링입니다.
연관키워드: 데이터 수집, 웹 스크래핑, 검색 엔진

Word2Vec

정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.
연관키워드: 자연어 처리, 임베딩, 의미론적 유사성
LLM (Large Language Model)
--------------------------------------------------------------------------------
문서 2
Token

정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다.
예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다.
연관키워드: 토큰화, 자연어 처리, 구문 분석

Tokenizer

정의: 토크나이저는 텍스트 데이터를 토큰으로 분할하는 도구입니다. 이는 자연어 처리에서 데이터를 전처리하는 데 사용됩니다.
예시: "I love programming."이라는 문장을 ["I", "love", "programming", "."]으로 분할합니다.
연관키워드: 토큰화, 자연어 처리, 구문 분석

VectorStore

정의: 벡터스토어는 벡터 형식으로 변환된 데이터를 저장하는 시스템입니다. 이는 검색, 분류 및 기타 데이터 분석 작업에 사용됩니다.
예시: 단어 임베딩 벡터들을 데이터베이스에 저장하여 빠르게 접근할 수 있습니다.
연관키워드: 임베딩, 데이터베이스, 벡터화

SQL
-------------------------